# Проект: Прогнозирование температуры звезды

## Описание проекта

Задача от обсерватории «Небо на ладони»: 

С помощью нейросети определять температуру на поверхности обнаруженных звёзд. Обычно для расчёта температуры учёные пользуются следующими методами:

- Закон смещения Вина.
- Закон Стефана-Больцмана.
- Спектральный анализ.

Каждый из них имеет плюсы и минусы. Обсерватория хочет внедрить технологии машинного обучения для предсказания температуры звёзд, надеясь, что этот метод будет наиболее точным и удобным.

В базе обсерватории есть характеристики уже изученных 240 звёзд.

## Настройка окружения

### Импорты

In [ ]:
%pip install -q phik
%pip install -q optuna
%pip install -q python-Levenshtein
%pip install -Uq scikit-learn

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from phik import phik_matrix

import plotly.express as px
import plotly.graph_objects as go

import optuna

import seaborn as sns
import math

import warnings

from Levenshtein import distance
from collections import defaultdict, Counter

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error

import random
import torch
import torch.nn as nn

### Настройки отображения

In [ ]:
# output settings
warnings.filterwarnings("ignore")

options = {
    "display.max_rows": None,
    "display.max_columns": None,
    "display.float_format": "{:,.2f}".format,
    "display.max_colwidth": None,
}

# Применяем опции через цикл
for option, value in options.items():
    pd.set_option(option, value)

### Объявление функций

In [ ]:
def check_size(current_df: pd.DataFrame, original_df: pd.DataFrame):
    print("Количество записей в текущем датасете: {}".format(len(current_df)))
    print("Количество записей в оригинальном датасете: {}".format(len(original_df)))
    print("Процент от начального объема данных: {:.2%}".format(len(current_df) / len(original_df)))

In [ ]:
HOST = "https://code.s3.yandex.net"
HTTP_PREFIX = "http"


def get_dataset_path(dataset_path: str):
    # check server request --> relative path --> absolute path --> yandex server request
    path = (
        dataset_path
        if dataset_path.startswith(HTTP_PREFIX)
        else "." + dataset_path if os.path.exists("." + dataset_path)
        else dataset_path if os.path.exists(dataset_path)
        else HOST + dataset_path
    )
    print("Dataset path:", path)
    return path


# load csv
def load_csv(dataset_path: str, **kwargs):
    path = get_dataset_path(dataset_path)
    try:
        return pd.read_csv(filepath_or_buffer=path, **kwargs)
    except Exception as ex:
        print("Could not load csv. Exception:", str(ex))

In [ ]:
def describe_dataset(df):
    """
    Выводит общую информацию и статистику по датасету pandas.

    Параметры:
    - df (pandas.DataFrame): Исходный датасет.
    """
    # 1. Общая информация о датасете
    print("*** Общая информация ***\n")
    print(df.info())
    print("\n")

    # 2. Статистика по количественным данным
    print("*** Статистика по количественным данным ***")
    display(df.describe())
    print("\n")

   # 3. Статистика по категориальным данным
    print("*** Статистика по категориальным данным ***")
    category_cols = df.select_dtypes(include=['object', 'category']).columns
    if len(category_cols) > 0:
        display(df[category_cols].describe())
    else:
        print("Нет категориальных признаков для анализа.")
    print("\n")

    # 4. Информация по колонкам
    print("*** Информация по колонкам ***")
    column_info = pd.DataFrame(
        {
            "type": df.dtypes,
            "na_count": df.isna().sum(),
            "empty_count": (df == "").sum(),
            "unique_count": df.nunique(),
        }
    )
    display(column_info)
    print("\n")

    # 5. Количество дубликатов
    duplicates = df.duplicated().sum()
    print(f"*** Количество дубликатов: {duplicates} ***")
    print("\n")

    # 6. Первые 5 строк датасета
    print("*** Первые 5 строк датасета ***")
    display(df.head())

## Загрузка данных

Проведем первичную (предварительную) загрузку небольшого количества строк данных датасета, чтобы посмотреть на данные и оценить их

In [ ]:
df_dict = {
    "df_stars": "/datasets/6_class.csv",
}

for df_name, df_file_path in df_dict.items():
    name = df_name
    print("-" * 10, f"start {name}", "-" * 10, "\n")
    path = get_dataset_path(df_file_path)
    df_name = pd.read_csv(path, nrows=5)
    display(df_name.head())
    display(df_name.dtypes)
    print("-" * 10, f"end {name}", "-" * 10, "\n")

Промежуточные выводы:
- первый столбец -- это индекс записи
- имена колонок можно перевести в нижний регистр
- поля Star type и Star color являются категориями, изменим им тип на category

In [ ]:
df_stars_original = load_csv("/datasets/6_class.csv", index_col=0, dtype={"Star type": "category", "Star color": "category"})
# переименование столбцов
cleaned_columns = {col: col.split("(")[0].strip().replace(" ", "_").lower() for col in df_stars_original.columns}
df_stars_original = df_stars_original.rename(columns=cleaned_columns)

df_stars = df_stars_original.copy()
display(df_stars.head())
display(df_stars.dtypes)

### Промежуточные выводы

Данные загружены, выглядят корректно, имена колонок и типы признаков правильные

## Предобработка и анализ данных

### Первичный анализ данных

In [ ]:
df_dict = {
    "df_stars": df_stars,
}

In [ ]:
for df_name, df in df_dict.items():
    print(f"{'-' * 10} start: {df_name} {'-' * 10}\n")
    describe_dataset(df)
    print(f"\n{'-' * 10} end: {df_name} {'-' * 10}\n")

Найдем все значения признака star_color

In [ ]:
print("star_color:\n", df_stars["star_color"].unique().tolist())

Промежуточные выводы:

- Данные не содержат пропусков
- Явных дубликатов нет
- Неявные дубликаты в значениях признака `star_color`
- Признаки `luminosity` и `radius` содержат подозрительно маленькие и большие значения, а также подозрительно большие значения стандратного отклонения. Возможно, выбросы или ошибки в данных

### Неявные дубликаты

Функции для проверки наличия наявных дубликатов в категориальных признаках.

In [ ]:
def normalize_string(s):
    # Удаляем лишнюю пунктуацию и пробелы, приводим к нижнему регистру
    s = ''.join(c for c in s if c.isalnum() or c.isspace()).strip().lower()
    return s


def find_implicit_duplicates(dataframe):
    results = []

    category_cols = dataframe.select_dtypes(include=['category']).columns
    
    for column_name in category_cols:
        values = dataframe[column_name].values.tolist()
        
        # Создаем словарь соответствия оригинальных значений и их нормализованных версий
        normalized_values = {v: normalize_string(v) for v in values}
        
        # Сохраняем оригинальные значения, соответствующие каждой нормализованной форме
        inverse_mapping = defaultdict(list)
        for orig_val, norm_val in normalized_values.items():
            inverse_mapping[norm_val].append(orig_val)
        
        # Уникальные нормализованные значения
        unique_normalized_values = list(inverse_mapping.keys())
        
        # Создание групп похожих значений
        similar_groups = defaultdict(list)
        
        # Объединяем похожие нормализованные значения в одну группу
        for norm_val in unique_normalized_values:
            found_group = False
            for existing_group in similar_groups:
                dist = distance(norm_val, existing_group)
                if dist <= len(existing_group) * 0.2:  # Расстояние <= 20%
                    similar_groups[existing_group].extend(inverse_mapping[norm_val])
                    found_group = True
                    break
            if not found_group:
                similar_groups[norm_val] = inverse_mapping[norm_val]
        
        # Формируем итоговый отчет
        result_column = f'{"-" * 10} Столбец: {column_name} {"-" * 10}\n\n'
        has_duplicates = False
        for group_key, group_members in sorted(similar_groups.items(), key=lambda x: len(x[1]), reverse=True):
            # Выбор самого популярного оригинального значения
            most_common_value = Counter(group_members).most_common(1)[0][0]
            similar_values = [v for v in group_members if v != most_common_value]
            
            if similar_values:
                has_duplicates = True
                result_column += f'- Наиболее частое значение: {most_common_value}\nПодобные значения: {similar_values}\n\n'
        
        if not has_duplicates:
            result_column += '- Неявных дубликатов не обнаружено.\n'
        
        results.append(result_column)
    
    return '\n'.join(results)

In [ ]:
print(find_implicit_duplicates(df_stars))

Заменим дубликаты на наиболее часто встречающееся значение и приведем к нижнему регистру

In [ ]:
df_stars['star_color'] = df_stars['star_color'].replace(['Blue white', 'Blue white ', 'Blue-white', 'Blue-White'], 'Blue White')
df_stars['star_color'] = df_stars['star_color'].replace(['white'], 'White')
df_stars['star_color'] = df_stars['star_color'].replace(['Blue '], 'Blue')
df_stars['star_color'] = df_stars['star_color'].replace(['Yellowish'], 'yellowish')

In [ ]:
df_stars['star_color'] = df_stars['star_color'].str.lower().astype('category')

In [ ]:
print(find_implicit_duplicates(df_stars))

Проверим значения признака star_color

In [ ]:
print("star_color:\n", df_stars["star_color"].unique().tolist())

- Заменим white-yellow на yellow-white
- Заменим yellowish white на yellow white
- Заменим yellowish на yellow
- Заменим whitish на white
- Заменим pale yellow orange на yellow orange
- Заменим дефисы на пробелы

In [ ]:
df_stars['star_color'] = df_stars['star_color'].replace(['white-yellow'], 'yellow-white')
df_stars['star_color'] = df_stars['star_color'].replace(['yellowish white'], 'yellow white')
df_stars['star_color'] = df_stars['star_color'].replace(['yellowish'], 'yellow')
df_stars['star_color'] = df_stars['star_color'].replace(['whitish'], 'white')
df_stars['star_color'] = df_stars['star_color'].replace(['pale yellow orange', 'orange-red'], 'orange')

In [ ]:
df_stars['star_color'] = df_stars['star_color'].str.replace('-', ' ').astype('category')

In [ ]:
print("star_color:\n", df_stars["star_color"].unique().tolist())

Неявные дубликаты устранены

### Явные дубликаты

Проверим наличие явных дубликатов

In [ ]:
def detect_duplicates(df):
    """
    Выводит общее количество дубликатов, первые 5 строк с дубликатами или сообщение "Явных дубликатов не обнаружено".
    
    Параметры:
    - df (pandas.DataFrame): Исходный датасет.
    """
    # Поиск дубликатов
    duplicated_rows = df[df.duplicated()]

    # Вывод результата
    if len(duplicated_rows) > 0:
        print(f"Общее количество дубликатов: {len(duplicated_rows)}")
        display(duplicated_rows.head())
    else:
        print("Явных дубликатов не обнаружено.")

In [ ]:
df_dict = {
    "df_stars": df_stars,
}

for df_name, df in df_dict.items():
    print(f"{'-' * 10} start: {df_name} {'-' * 10}\n")
    detect_duplicates(df)
    print(f"\n{'-' * 10} end: {df_name} {'-' * 10}\n")

Проверим полноту данных

In [ ]:
check_size(df_stars, df_stars_original)

### Промежуточные выводы

В ходе предварительной обработки данных было сделано следующее:
- Проведена проверка на наличие пропусков в данных. Пропусков не обнаружено.
- Проведена проверка на наличие неявных дубликатов. Неявные дубликаты в категориальных признаках устранены.
- Проведена проверка на наличие явных дубликатов, явных дубликатов не обнаружено.

## Исследовательский анализ данных

### Количественные признаки

Создадим функцию для визуализации количественных признаков. Фукнция будет строить график распределения и размаха признака.

In [ ]:
def visualize_numerical_distribution_and_range(df, bins_rule="sturges", numeric_cols=None):
    """
    Рисует графики распределения (гистограмма) и размаха (коробчатая диаграмма) для всех числовых признаков.

    Параметры:
    - df (pandas.DataFrame): Исходный датасет.
    - bins_rule (str): Правило для автоматического подбора количества корзин. Возможные значения:
                     'sturges' (правило Стерджеса) или 'sqrt' (корень квадратный).
    """
    # Отбираем только числовые столбцы
    if numeric_cols is None or not list(numeric_cols):
        numeric_cols = df.select_dtypes(include=["number"]).columns

    # Функция для автоматического подбора количества корзин
    def auto_binning(data_length, rule=bins_rule):
        if rule == "sturges":
            return int(math.ceil(math.log2(data_length))) + 1
        elif rule == "sqrt":
            return int(math.sqrt(data_length))
        else:
            raise ValueError("Invalid binning rule provided.")

    # Проходим по каждому числовому столбцу и строим графики
    for col in numeric_cols:
        # Подбор оптимального количества корзин
        optimal_bins = auto_binning(len(df))

        # Гистограмма с боксплотом (график распределения и размаха)
        fig = px.histogram(
            df,
            x=col,
            marginal="box",
            barmode="group",
            nbins=optimal_bins,
            title=f'Распределение и размах признака "{col}"',
        )

        # Визуализация графика
        fig.update_layout(bargap=0.02)
        fig.show()

In [ ]:
numeric_columns = df_stars.select_dtypes(include=['number']).columns
numeric_columns

Построим графики распределения и размаха для количественных признаков.

In [ ]:
visualize_numerical_distribution_and_range(df_stars, "sqrt", numeric_columns)

#### temperature

Для признака `temperature` наблюдается большое количество значений ниже 5000К

---
Температура на поверхности звезды (точнее — в её фотосфере) измеряется в кельвинах (К) и варьируется в очень широких пределах — от ~2 000 К у самых холодных звёзд до ~50 000–60 000 К и выше у самых горячих.

Особые типы звёзд и их температуры

Красные гиганты и сверхгиганты:
Обычно T≈3000–5000 К (класс K–M), несмотря на огромную светимость.
Пример: Антарес (α Скорпиона) — ∼3660 К.

Белые карлики:
Молодые белые карлики могут иметь T>100000 К.
Старые — остывают до T∼4000–10000 К.
Пример: Сириус B — ∼25000 К.

Коричневые карлики (субзвёздные объекты):
T≲2000 К (иногда ниже 1000 К).
Пример: WISE 1828+2650 — ∼300–400 К.

Звёзды Вольфа‑Райе (очень горячие и массивные):
T≳30000 К (до ∼200000 К у некоторых).
Пример: WR 124 — ∼50000 К.

Возможные значения температуры поверхности звёзд:
- Минимум: ∼300–400 К (самые холодные коричневые карлики).
- Максимум: ≳50000–60000 К (звёзды классов O и Вольфа‑Райе; у отдельных объектов — до ∼200000 К).
- Типичный диапазон для «обычных» звёзд (главная последовательность): 2000–60000 К.
- Солнце: ∼5772 К.

Таким образом, в диапазон <5000K должны попадать преимущественно звезды с типом 0, 1, 5.

Проверим эту теорию.

In [ ]:
df_stars_temperature_5k = df_stars.query("temperature < 5000 and star_type not in ('0', '1', '5')")
len(df_stars_temperature_5k)

Всего 12 звезд других типов. Найдем их типы и количество по типам.

In [ ]:
counts = df_stars_temperature_5k["star_type"].value_counts()
non_zero_counts = counts[counts > 0]
print(non_zero_counts)

9 сверхгигантов и 3 звезды главной последовательности. Оба типа выглядят вполне возможными для указанных значений.

#### luminosity

Для признака `luminosity` наблюдается большое количество значенией близких к 0.

---
Светимость звезды относительно Солнца может принимать очень широкий диапазон значений в зависимости от типа звезды и этапа её эволюции.

По порядку величины светимость звёзд варьируется от $10^{−4}$ $L_0$​ до $10^{6}$ $L_0$:

Очень тусклые звёзды (красные карлики, поздние спектральные классы):
L ≈ $10^{−4}$ $L_0$​ – $10^{−2}$ $L_0$.
Пример: звёзды типа M‑карликов (например, Проксима Центавра — около 0,0017 $L_0$).

Звёзды солнечной массы и спектрального класса G:
L ≈ 0,1 $L_0$ – 10 $L_0$.
Пример: Солнце — ровно 1 $L_0$​; Альфа Центавра — около 1,52 $L_0$.

Яркие звёзды средних и больших масс (классы A, F, ранние G):
L ≈ 10 $L_0$ – $10^{3}$ $L_0$.
Пример: Сириус — около 25,4 $L_0$​; Вега — около 40 $L_0$.

Сверхгиганты и яркие гиганты (классы O, B, ранние A):
L ≈ $10^{3}$ $L_0$ – $10^{6}$ $L_0$.
Пример: Ригель (B‑сверхгигант) — около 120000 $L_0$​; Дзета Кормы (O‑сверхгигант) — порядка 800000 $L_0$.

Экстремально яркие звёзды (например, звёзды Вольфа‑Райе, гипергиганты):
L может превышать $10^{6}$ $L_0$.
Пример: R136a1 (звезда Вольфа‑Райе в скоплении R136) — свыше 6100000 $L_0$.

Возможные значения $L/L_0$​ для звёзд:
- Минимум: ~$10^{-4}$ $L_0$ (очень холодные красные карлики).
- Максимум: $10^{6}$ $L_0$​ (самые массивные сверхгиганты и звёзды Вольфа‑Райе).
- Типичный диапазон для наблюдаемых звёзд: от 0,0001 до нескольких сотен тысяч $L_0$.

Таким образом, в этот диапазон (<50K) согласно спецификации и приведенной выше справки попадают звезды с типами 0, 1, 2, 3.

Проверим эту теорию.

In [ ]:
df_stars_luminosity_50k = df_stars.query("luminosity < 50000")
df_stars_luminosity_50k["star_type"].unique().tolist()

Найдем минимальное значение светимости для этих звезд

In [ ]:
min(df_stars_luminosity_50k["luminosity"])

Найдем минимальное значение светимости для звезд типов 4 и 5.

In [ ]:
df_stars_star_type_45 = df_stars.query("star_type in ('4', '5')")
min(df_stars_star_type_45["luminosity"])

Данные в целом выглядят корректно.

#### radius

Для признака `radius` наблюдается подавляющее количество значений (200 значений) меньше 100.

---
Радиус звезды относительно радиуса Солнца может принимать крайне широкий диапазон значений -- от крошечных нейтронных звёзд до исполинских сверхгигантов.

Типичные диапазоны (по типам звёзд)

Нейтронные звёзды: 
R≈10км → $1,4⋅10^{−5}$ (в 70 000 раз меньше Солнца).
Это компактные остатки взрывов сверхновых

Белые карлики:
R≈0,01 – 0,02 (примерно как Земля).
Пример: Сириус B — около 0,008.

Красные карлики (M‑класс):
R≈0,1 – 0,6.
Пример: Проксима Центавра — около 0,15.

Звёзды главной последовательности (G‑, F‑, A‑классы):
R≈0,6 – 2.
Пример: Солнце — ровно 1; Сириус A — около 1,7.

Гиганты (поздние стадии эволюции):
R≈10 – 100.
Пример: Арктур (K‑гигант) — около 25,4; Поллукс (K‑гигант) — около 8,8.

Сверхгиганты и гипергиганты (O‑, B‑, K‑, M‑классы):
R≈100​ – 2000​ и более.

Возможные значения:
- Минимум: ∼$1,4⋅10^{−5}$ (нейтронные звёзды).
- Максимум: >2000 (красные гипергиганты).
- Типичный диапазон для наблюдаемых звёзд: от 0,008 (белые карлики) до 1000–2000 (сверхгиганты).

Таким образом, в диапазон >100 согласно спецификации и приведенной выше справки попадают звезды с типом 5.

Проверим эту теорию.

In [ ]:
df_stars_radius_more_100 = df_stars.query("radius > 100")
df_stars_radius_more_100["star_type"].unique().tolist()

Найдем мксимальное минимальное и максимальное значение радиуса для остальных типов звезд.

In [ ]:
df_stars_radius_less_100 = df_stars.query("star_type != '5'")
print(min(df_stars_radius_less_100["radius"]))
print(max(df_stars_radius_less_100["radius"]))

Найдем также минимальное значение радиуса для звезд типа 5

In [ ]:
df_stars_radius_star_type_5 = df_stars.query("star_type == '5'")
print(min(df_stars_radius_star_type_5["radius"]))

Данные в целом выглядят корректно.

#### absolute_magnitude

---
Звёздная величина `Блеск Звезды` может принимать значения:

Видимая (m): от −26,7 m (Солнце) до +31,5 m (самые слабые объекты, наблюдаемые «Хабблом»).

Абсолютная (M): у звёзд — примерно от −10 m (сверхгиганты) до +17 m (красные карлики); у галактик может быть ещё ниже (например, Галактика Андромеды: M ≈ −20,5 m).

Особенности:
- Отрицательные значения: у очень ярких объектов (Солнце, Луна, планеты, ярчайшие звёзды).
- Положительные значения: у слабых объектов (тусклые звёзды, далёкие галактики).
- Зависимость от диапазона: звёздная величина измеряется в разных спектральных полосах (например, V — визуальный, B — синий, R — красный). Величина может заметно различаться в разных фильтрах.
- Переменность: у пульсирующих, затменных и вспыхивающих звёзд m меняется со временем.

Согласно спецификации по типам звезд и приведенной выше справки можно сделать вывод, что блеск звезды со значениями >10 принадлежит всем видам карликов (типы 0, 1, 2).

Проверим эту теорию.

In [ ]:
df_stars_magnitude_more_10 = df_stars.query("absolute_magnitude > 10")
df_stars_magnitude_more_10["star_type"].unique().tolist()

Найдем минимальное значение блеска звезды для всех видов карликов

In [ ]:
df_stars_magnitude_star_type_012 = df_stars.query("star_type in ('0', '1', '2')")
min(df_stars_magnitude_star_type_012["absolute_magnitude"])

Найдем минимальное и максимальное значения блеска для остальных типов звезд

In [ ]:
df_stars_magnitude_star_type_345 = df_stars.query("star_type not in ('0', '1', '2')")
print(min(df_stars_magnitude_star_type_345["absolute_magnitude"]))
print(max(df_stars_magnitude_star_type_345["absolute_magnitude"]))

Данные в целом выглядят корректно.

### Категориальные признаки

Создадим функцию для визуализации категориальных признаков. Фукнция будет строить столбчатую и круговую диаграммы для признака.

In [ ]:
def visualize_categorical_columns(df, category_cols=None):
    """
    Строит столбчатые и круговые диаграммы для категориальных признаков датасета.
    
    Параметры:
    - df (pandas.DataFrame): Исходный датасет.
    """
    # Получаем список категориальных столбцов
    if category_cols is None or not list(category_cols):
        category_cols = df.select_dtypes(exclude=['number']).columns
    
    # Для каждого категориального столбца строим диаграммы
    for col in category_cols:
        # Частотная таблица для столбца
        freq_table = df[col].value_counts()
        
        # Бар-чарт (столбчатая диаграмма)
        fig_bar = px.bar(freq_table, x=freq_table.index, y=freq_table.values, 
                         labels={"x": f"{col}", "y": "Частота"},
                         title=f"Бар-диаграмма для {col}")
        fig_bar.show()
        
        # Круговая диаграмма (pie chart)
        fig_pie = px.pie(names=freq_table.index, values=freq_table.values, 
                         title=f"Круговая диаграмма для {col}")
        fig_pie.show()

In [ ]:
category_columns = df_stars.select_dtypes(include=['category']).columns
category_columns

Построим графики для категориальных признаков

In [ ]:
visualize_categorical_columns(df_stars, category_columns)

- Данные поровну разбиты по типу звезд.
- Звезды красного цвета встречаются чаще других. На втором месте звезды с голубым цветом.
- Звезды желтого (3 наблюдения) и оранжевого (4 наблюдения) цветов встречаются реже всего. Возможно, потребуется применение аугментации данных, чтобы лучше предсказывать значение температуры для них.
- Столбчатая диаграмма по типу звезд (количество) косвенно подтверждает выводы, сделанные на этапе анализа количественных признаков.

### Диаграммы рассеяния

Создадим функцию для построения диаграммы рассеяния (scatter plot) переданного количественного признака в зависимости от значения переданного категориого признака.

In [ ]:
def scatter_plot_by_category(df, quant_col, cat_col):
    """
    Отображает диаграмму рассеяния (scatter plot) количественного признака относительно категориального.
    
    Параметры:
    - df: исходный Dataset (pandas DataFrame).
    - quant_col: название количественного признака (числового типа).
    - cat_col: название категориального признака (строкового типа или факторизированного).
    """
    # Проверяем наличие указанных столбцов в DataFrame
    if quant_col not in df.columns or cat_col not in df.columns:
        raise ValueError(f"В указанном датасете отсутствуют указанные столбцы '{quant_col}', '{cat_col}'.")
        
    # Строим диаграмму рассеяния с группировкой по категориям
    fig = px.scatter(
        data_frame=df,
        x=quant_col, # Количественный признак по оси X
        y=df.index, # Используем индекс строки как координату Y
        color=cat_col, # Цвет точек определяется категорией
        hover_data=[df.index], # Подсказка при наведении мыши включает номер строки
        title=f"{quant_col} vs {cat_col}",
        # width=800, height=600 # Размер графика
    )
    
    # Обновляем разметку осей и заголовков
    fig.update_xaxes(title_text=quant_col)
    fig.update_yaxes(title_text="Наблюдения")
    fig.update_traces(marker=dict(size=8)) # Размер маркера точки
    
    return fig.show()

Построим диаграмму рассеяния для количественных признаков в зависимости от типа звезды.

In [ ]:
for num_col in numeric_columns:
    scatter_plot_by_category(df_stars, num_col, "star_type")

Полученные результаты подтверждают выводы, сделанные на предыдущих шагах.

### Промежуточные выводы

Был проведен исследовательский анализ количественных и категориальных признаков датасета. Построены диаграммы рассеяния для количественных признаков в записимости от признака `star_type`. Данные в целом выглядят корректно. Дополнительные преобразования данных не производились.

## Корреляционный анализ

Напишем функции для проведения корреля корреляционного анализа данных.

In [ ]:
def correlation_heatmap(df, interval_cols=None):
    """
    Строит матрицу корреляции с использованием phik_matrix и выводит тепловую карту с аннотациями.
    
    Параметры:
    - df (pandas.DataFrame): Исходный датасет.
    """
    # Отбираем только числовые столбцы
    if interval_cols is None or not list(interval_cols):
        interval_cols = df.select_dtypes(include=["number"]).columns

    # Вычисляем матрицу корреляции Phik
    corr_matrix = phik_matrix(df, interval_cols=interval_cols, verbose=False)

    # Определяем оптимальный размер фигуры
    num_cols = len(corr_matrix.columns)
    size_per_col = 0.7  # размер графика на колонку
    figure_width = num_cols * size_per_col
    figure_height = num_cols * size_per_col

    # Вывод тепловой карты
    plt.figure(figsize=(figure_width, figure_height))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm')
    plt.title("Матрица корреляции Phik")
    plt.show()

Построим матрицу корреляции признаков.

In [ ]:
numeric_columns = df_stars.select_dtypes(include=["number"]).columns

correlation_heatmap(df_stars)

Определим степень зависимости целевого признака температуры от остальных признаков

In [ ]:
def correlation_analysis(corr_matrix, target_column):
    """
    Анализирует матрицу корреляции и выводит степень зависимости целевого признака от остальных признаков.
    
    Параметры:
    - corr_matrix (pandas.DataFrame): Матрица корреляции.
    - target_column (str): Название целевого признака.
    """
    # Получаем строку с корреляциями целевого признака
    correlations = round(corr_matrix[target_column].abs(), 2)
    
    # Исключаем сам целевой признак
    correlations = correlations.drop(target_column)
    
    # Классифицируем зависимости согласно таблице
    very_low_dependency = correlations[(correlations >= 0) & (correlations < 0.2)].index.tolist()
    low_dependency = correlations[(correlations >= 0.2) & (correlations < 0.5)].index.tolist()
    medium_dependency = correlations[(correlations >= 0.5) & (correlations < 0.7)].index.tolist()
    high_dependency = correlations[(correlations >= 0.7) & (correlations < 0.9)].index.tolist()
    very_high_dependency = correlations[(correlations >= 0.9) & (correlations <= 1)].index.tolist()
    
    # Вывод результатов
    print(f"Признак {target_column} имеет:")
    if very_high_dependency:
        print(f"- очень высокую зависимость (> 0.9) от признаков {very_high_dependency}")
    if high_dependency:
        print(f"- высокую зависимость (0.7-0.9) от признаков {high_dependency}")
    if medium_dependency:
        print(f"- среднюю зависимость (0.5-0.7) от признаков {medium_dependency}")
    if low_dependency:
        print(f"- слабую зависимость (0.2-0.5) от признаков {low_dependency}")
    if very_low_dependency:
        print(f"- очень слабую зависимость (0-0.2) от признаков {very_low_dependency}")
    if not (very_high_dependency or high_dependency or medium_dependency or low_dependency or very_low_dependency):
        print("- признаков с достаточной степенью зависимости не найдено.")

In [ ]:
corr_matrix = phik_matrix(df_stars, verbose=False)

correlation_analysis(corr_matrix, "temperature")

### Промежуточные выводы

Матрица корреляции показала следующие зависимости признака `temperature`:
- высокую зависимость (0.7-0.9) от признаков ['absolute_magnitude', 'star_color']
- среднюю зависимость (0.5-0.7) от признаков ['luminosity', 'star_type']
- слабую зависимость (0.2-0.5) от признаков ['radius']

## Подготовка данных к построению модели

### Подготовка обучающей и тестовой выборок

Подготовим выборки: обучающую и тестовую

In [ ]:
RANDOM_STATE = 1
TEST_SIZE = 0.25

X = df_stars.drop(columns=["temperature"])
y = df_stars["temperature"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    shuffle=True,
    stratify=df_stars["star_color"],
    random_state=RANDOM_STATE,
)

print(X.shape, y.shape)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

### Масштабирование количественных данных

Так как имеются признаки с большим динамическим диапазоном и вероятностью наличия экстремальных значений, обычные подходы вроде StandardScaler` или `MinMaxScaler` могут плохо справляться с такими признаками, потому что они будут искажать различия между большинством объектов из-за влияния крайних точек.

Поэтому рекомендуется использовать два подхода:
- Робастный скейлер (`RobustScaler`)Хороший выбор, если есть вероятность выбросов. `RobustScaler` устойчив к влиянию анормальных значений благодаря расчету межквартильного расстояния (IQR). Такой подход сохраняет распределение большинства объектов даже при наличии редких крупных экстремумов.
- Log-преобразование + `MinMaxScaler` или `StandardScaler`. Иногда помогает применение логарифмического преобразования для сглаживания разницы между малыми и большими значениями. После логарифмического преобразования можно использовать `MinMaxScaler` или `StandardScaler`, чтобы ограничить диапазон и центрировать признаки вокруг нуля.

Будем использовать `RobustScaler` для масштабирования количественных признаков. Категориальные признаки обработаем с помощью `OneHotEncoder`.

In [ ]:
# Выделим количественные и категоривальные признаки
num_cols = X_train.select_dtypes(include=['number']).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=['number']).columns.tolist()

print(f"Количественные признаки: {num_cols}")
print(f"Категориальные признаки: {cat_cols}")

Несмотря на то, что признак `star_type` представлен целым числом, его также имеет смысл закодировать с использованием `OneHotEncoder`, чтобы не вводить сеть в заблуждение относительно "близости" представленных значений.

Проведем предобработку признаков с использованием `ColumnTransformer`.

In [ ]:
# Создадим transformer, который применяет разные препроцессоры к разным признакам
preprocessor = ColumnTransformer(
    transformers=[
        ('num', RobustScaler(), num_cols),  # Масштабируем количественные признаки
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_cols),  # Кодируем категориальные признаки
    ],
    remainder='passthrough'  # Оставляем остальные признаки без изменений
)

# Применим preprocessor к тренировочному набору
X_train_preprocessed = preprocessor.fit_transform(X_train)

# Применим preprocessor к тестовому набору
X_test_preprocessed = preprocessor.transform(X_test)

Преобразуем обработанные данные в тензоры

In [ ]:
X_train_tensor = torch.FloatTensor(X_train_preprocessed)
X_test_tensor = torch.FloatTensor(X_test_preprocessed)
y_train_tensor = torch.FloatTensor(y_train.values)
y_test_tensor = torch.FloatTensor(y_test.values)

Данные подготовлены к построению модели нейронной сети

### Промежуточные выводы

- Данные были разбиты на обучающиую и тестовую выборки
- Проведено масштабирование количественных признаков с использованием `RobustScaler`, категориальные признаки обработаны с помощью `OneHotEncoder`

## Построение базовой нейронной сети

Получим количество признаков после проведения их масштабирования и кодирования. Это будет количество нейронов входного слоя нейронной сети.

In [ ]:
# Получим имена колонок после обработки
feature_names = preprocessor.get_feature_names_out()
print("feature_names:", feature_names)
print("Количество признаков:", len(feature_names))

Для задачи линейной регрессии с набором данных, состоящим из 16 признаков, выберем следующую архитектуру нейронной сети:

- Входной слой: 16 нейронов, соответствует количеству признаков в датасете после проведения масштабирования и кодирования
- Скрытый слой: количество нейронов — около половины входных признаков. Так как их 16, возьмем половину — 8 нейронов
- Выходной слой: для задачи линейной регрессии выходной слой содержит один нейрон, т.к. предсказываем одну целевую переменную.
- Функция активации: возьмем ReLU, так как она ускоряет процесс обучения и помогает избежать проблемы исчезновения градиентов.

Будем использовать оптимизатор Adam для автоматической адаптации скорости обучения. В качестве функции потерь будем использовать MSE, так как она подходит для задач регрессии и легко интерпретируется, а RMSE будем использовать в качестве метрики оценки полученной модели.

In [ ]:
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.use_deterministic_algorithms(True)

In [ ]:
n_in_neurons = 16
n_hidden_neurons = 8
n_out_neurons = 1

learn_rate = 0.1
# learn_rate = 1e-3

num_epochs = 25000

net = nn.Sequential(
    nn.Linear(n_in_neurons, n_hidden_neurons),
    nn.ReLU(),
    nn.Linear(n_hidden_neurons, n_out_neurons),
    nn.ReLU(),
)

# alternative

# n_hidden_neurons_1 = 8
# n_hidden_neurons_2 = 4
# 
# net = nn.Sequential(
#     nn.Linear(n_in_neurons, n_hidden_neurons_1),
#     nn.ReLU(),
#     nn.Linear(n_hidden_neurons_1, n_hidden_neurons_2),
#     nn.ReLU(),
#     nn.Linear(n_hidden_neurons_2, n_out_neurons),
#     nn.ReLU(),
# )

optimizer = torch.optim.Adam(net.parameters(), lr=learn_rate)
loss = nn.MSELoss()

In [ ]:
for epoch in range(num_epochs):
    optimizer.zero_grad()

    preds_train = net.forward(X_train_tensor).flatten()

    loss_value = loss(preds_train, y_train_tensor)
    loss_value.backward()

    optimizer.step()

    if epoch % 100 == 0 or epoch == num_epochs - 1:
        net.eval()  # перевод сети в режим предсказания
        preds_train = net.forward(X_train_tensor).flatten()
        mse_train = loss(preds_train, y_train_tensor)
        rmse_train = torch.sqrt(mse_train)
        print(f'Epoch [{epoch+1}/{num_epochs}] Train RMSE: {rmse_train:.4f}')

Финальная проверка на тестовых данных

In [ ]:
net.eval()
preds_test = net.forward(X_test_tensor).flatten()
mse_test = loss(preds_test, y_test_tensor)
rmse_test = torch.sqrt(mse_test)
print(f'Epoch [{epoch+1}/{num_epochs}] Test RMSE: {rmse_test:.4f}')

Построим график Факт-Прогноз для предсказаний модели. Для этого создадим функцию для отрисовки значений

In [ ]:
def show_compare_plot(pred: np.array, true: np.array):    
    indices = np.arange(len(true))

    # Создание объектов Bar для каждого типа данных
    fig = go.Figure(data=[
        go.Bar(name='Реальные', x=indices, y=true),
        go.Bar(name='Прогнозируемые', x=indices, y=pred)
    ])

    # Настройка вида графика
    fig.update_layout(barmode='group')
    fig.update_xaxes(title_text='Индексы')
    fig.update_yaxes(title_text='Значения')
    fig.update_layout(title_text='Сравнение реальных и прогнозных значений')

    fig.show()

Сравнение на тестовых данных

In [ ]:
show_compare_plot(preds_test.detach().numpy(), y_test_tensor.numpy())

### Промежуточные выводы

Была построена базовая нейронная сеть, состоящая из 3 слоев: входного (16 нейронов), скрытого (8 нейронов) и выходного (1 нейрон).

Использован оптимизатор Adam. В качестве функции потерь использована MSE, а для оценки качества модели использована метрика RMSE.

Метрика RMSE на тестовых данных превысила пороговое значение 4500К и составила 5453.6724.

Дополнительно был построен график Факт-Прогноз для сравнения полученных значений целевого признака Температура с реальными данными.

## Улучшение нейронной сети

Создадим решение с перебором параметров нейросети. 

Список параметров для перебора будет включать как минимум «dropout» и «размер батча». Архитектуру нейронной сети (количество слоёв, нейронов, вид функции активации) оставим без изменений, чтобы сравнить результат.

Будем использовать Optuna для оптимизации гиперпараметров нейронной сети.

Добавим слой dropout и создадим класс для передачи в конструктор значение для dropout

In [ ]:
class SimpleNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout_prob):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden_dim, output_dim),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.net(x)

Создадим функцию для перебора параметров

In [ ]:
def objective(trial):
    # Пространство поиска параметров
    dropout = trial.suggest_categorical("dropout", [0.0, 0.1, 0.2, 0.25, 0.5])
    num_epochs = trial.suggest_categorical("num_epochs", [10000])
    batch_size = trial.suggest_categorical("batch_size", [8, 16, 32, 64])
    learning_rate = trial.suggest_categorical("learning_rate", [0.001, 0.01, 0.1])

    # Модель
    model = SimpleNet(n_in_neurons, n_hidden_neurons, n_out_neurons, dropout)

    # Оптимизация и потеря
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss = nn.MSELoss()

    # Обучение
    for _ in range(num_epochs):
        order = np.random.permutation(len(X_train_tensor)) # случайная перестановка индексов объектов
        num_batches = math.ceil(len(X_train_tensor) / batch_size)
        for batch_idx in range(num_batches):
            # номер стартового объекта батча
            start_index = (
                batch_idx * batch_size
            )
            optimizer.zero_grad()

            # индексы объектов текущего обатча
            batch_indexes = order[
                start_index : start_index + batch_size
            ]

            X_batch = X_train_tensor[batch_indexes]
            y_batch = y_train_tensor[batch_indexes]

            preds = model.forward(X_batch).flatten()
            loss_value = loss(preds, y_batch)
            loss_value.backward()
            optimizer.step()
    
    # Проверка качества на тренировочном датасете
    model.eval()
    preds_train = model.forward(X_train_tensor).flatten()
    mse_train = loss(preds_train, y_train_tensor)
    rmse_train = torch.sqrt(mse_train)

    # минимизируем ошибку RMSE
    return rmse_train

Запустим перебор параметров с помощью Optuna

In [ ]:
model = optuna.create_study(direction="minimize", study_name="Neuron Network")
model.optimize(objective, n_trials=50)

print("\nBest Parameters:", model.best_params)
print("Best Value:", model.best_value)

Проведем финальную проверку на тестовой выборке

In [ ]:
dropout = 0.0
num_epochs = 10000
batch_size = 16
learning_rate = 0.1

# Модель
model = SimpleNet(n_in_neurons, n_hidden_neurons, n_out_neurons, dropout)

# Оптимизация и потеря
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
loss = nn.MSELoss()

# Обучение
for _ in range(num_epochs):
    order = np.random.permutation(len(X_train_tensor)) # случайная перестановка индексов объектов
    num_batches = math.ceil(len(X_train_tensor) / batch_size)
    for batch_idx in range(num_batches):
        # номер стартового объекта батча
        start_index = (
            batch_idx * batch_size
        )
        optimizer.zero_grad()

        # индексы объектов текущего обатча
        batch_indexes = order[
            start_index : start_index + batch_size
        ]

        X_batch = X_train_tensor[batch_indexes]
        y_batch = y_train_tensor[batch_indexes]

        preds = model.forward(X_batch).flatten()
        loss_value = loss(preds, y_batch)
        loss_value.backward()
        optimizer.step()

In [ ]:
model.eval()
preds_test = model.forward(X_test_tensor).flatten()
mse_test = loss(preds_test, y_test_tensor)
rmse_test = torch.sqrt(mse_test)
print(f'Epoch [{epoch+1}/{num_epochs}] Test RMSE: {rmse_test:.4f}')

Сравнение на тестовых данных

In [ ]:
show_compare_plot(preds_test.detach().numpy(), y_test_tensor.numpy())

Результат стал лучше. Метрика RMSE на тестовых данных составила 4658.5361. Но это значение все равно превышает пороговое значение 4500К.

Похоже, сеть переобучилась, и на тренировочных данных показывает результаты сильно лучше, чем на тестовых.

Воспользуемся методом аугментации данных, чтобы улучшить ее показатели.

### Аугментация данных

Сгенерируем и добавим в датасет записи для тех цветов звезд, количество которых составляет 5% и меньше.

In [ ]:
print(df_stars["star_color"].value_counts(normalize=True, ascending=True))

Найдем диапазон распределения температур для этих цветов ("orange", "yellow", "yellow white", "white"), чтобы понимать, какие значения могут быть разумными.

In [ ]:
color_filter = df_stars["star_color"].isin(["orange", "yellow", "yellow white", "white"])
print(df_stars[color_filter].groupby("star_color")["temperature"].agg(['min', 'max']).dropna())

Создадим новые записи для этих цветов

In [ ]:
# Считаем количество записей для каждого цвета
current_count = df_stars["star_color"].value_counts()

# Зафиксируем границы температур
temperature_bounds = {
    "orange": (3749, 7230),
    "white": (7220, 14732),
    "yellow": (4077, 4980),
    "yellow white": (5300, 12990)
}

colors = temperature_bounds.keys()

# Склад для новых записей
new_records = []

# Формируем целевой набор записей
target_count = 30

# Работаем только с цветами из colors
for color in colors:
    # Сколько записей имеется
    num_existing = current_count.get(color, 0)
    
    # Сколько нужно записать, чтобы получилось ровно target_count
    num_needed = target_count - num_existing
    
    # Если записей уже достаточно, продолжаем дальше
    if num_needed <= 0:
        continue
    
    # Извлекаем оригинальные записи для этого цвета
    original_rows = df_stars[df_stars["star_color"] == color]
    
    # Если оригинальных записей нет, берем первую запись и копируем её
    if len(original_rows) == 0:
        row_to_copy = df_stars.iloc[[0]]
        row_to_copy["star_color"] = color
    else:
        row_to_copy = original_rows.sample(1)
    
    # Заполняем недостающие записи, создавая копии оригинальной записи
    additional_rows = pd.concat([row_to_copy]*num_needed, ignore_index=True)
    
    # Генерация новых температур
    additional_rows["temperature"] = np.random.uniform(
        low=temperature_bounds[color][0],
        high=temperature_bounds[color][1],
        size=len(additional_rows)
    )
    
    # Добавляем новые записи
    new_records.append(additional_rows)

# Объединяем с первоначальным датасетом
df_stars_final_data = pd.concat([df_stars] + new_records, ignore_index=True)

# Смотрим результат
print(df_stars_final_data["star_color"].value_counts())

Заново разобъем на тренировочную и тестовую выборки, проведем масштабирование признаков и обучим нейронную сеть

In [ ]:
X = df_stars_final_data.drop(columns=["temperature"])
y = df_stars_final_data["temperature"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    shuffle=True,
    stratify=df_stars_final_data["star_color"],
    random_state=RANDOM_STATE,
)

print(X.shape, y.shape)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In [ ]:
# Применим preprocessor к тренировочному набору
X_train_preprocessed = preprocessor.fit_transform(X_train)

# Применим preprocessor к тестовому набору
X_test_preprocessed = preprocessor.transform(X_test)

In [ ]:
X_train_tensor = torch.FloatTensor(X_train_preprocessed)
X_test_tensor = torch.FloatTensor(X_test_preprocessed)
y_train_tensor = torch.FloatTensor(y_train.values)
y_test_tensor = torch.FloatTensor(y_test.values)

Создадим функцию для перебора параметров

In [ ]:
def objective(trial):
    # Пространство поиска параметров
    dropout = trial.suggest_categorical("dropout", [0.0, 0.25, 0.5])
    num_epochs = trial.suggest_categorical("num_epochs", [100, 1000, 10000])
    batch_size = trial.suggest_categorical("batch_size", [62, 123, 246])
    learning_rate = trial.suggest_categorical("learning_rate", [0.001, 0.01, 0.1])

    # Модель
    model = SimpleNet(n_in_neurons, n_hidden_neurons, n_out_neurons, dropout)

    # Оптимизация и потеря
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss = nn.MSELoss()

    # Обучение
    for _ in range(num_epochs):
        order = np.random.permutation(len(X_train_tensor)) # случайная перестановка индексов объектов
        num_batches = math.ceil(len(X_train_tensor) / batch_size)
        for batch_idx in range(num_batches):
            # номер стартового объекта батча
            start_index = (
                batch_idx * batch_size
            )
            optimizer.zero_grad()

            # индексы объектов текущего обатча
            batch_indexes = order[
                start_index : start_index + batch_size
            ]

            X_batch = X_train_tensor[batch_indexes]
            y_batch = y_train_tensor[batch_indexes]

            preds = model.forward(X_batch).flatten()
            loss_value = loss(preds, y_batch)
            loss_value.backward()
            optimizer.step()
    
    # Проверка качества на тренировочном датасете
    model.eval()
    preds_train = model.forward(X_train_tensor).flatten()
    mse_train = loss(preds_train, y_train_tensor)
    rmse_train = torch.sqrt(mse_train)

    # минимизируем ошибку RMSE
    return rmse_train

Запустим перебор параметров с помощью Optuna

In [ ]:
model = optuna.create_study(direction="minimize", study_name="Neuron Network")
model.optimize(objective, n_trials=50)

print("\nBest Parameters:", model.best_params)
print("Best Value:", model.best_value)

Проведем финальную проверку на тестовой выборке

In [ ]:
dropout = 0.0
num_epochs = 10000
batch_size = 246
learning_rate = 0.1

# Модель
model = SimpleNet(n_in_neurons, n_hidden_neurons, n_out_neurons, dropout)

# Оптимизация и потеря
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
loss = nn.MSELoss()

# Обучение
for _ in range(num_epochs):
    order = np.random.permutation(len(X_train_tensor)) # случайная перестановка индексов объектов
    num_batches = math.ceil(len(X_train_tensor) / batch_size)
    for batch_idx in range(num_batches):
        # номер стартового объекта батча
        start_index = (
            batch_idx * batch_size
        )
        optimizer.zero_grad()

        # индексы объектов текущего обатча
        batch_indexes = order[
            start_index : start_index + batch_size
        ]

        X_batch = X_train_tensor[batch_indexes]
        y_batch = y_train_tensor[batch_indexes]

        preds = model.forward(X_batch).flatten()
        loss_value = loss(preds, y_batch)
        loss_value.backward()
        optimizer.step()

In [ ]:
model.eval()
preds_test = model.forward(X_test_tensor).flatten()
mse_test = loss(preds_test, y_test_tensor)
rmse_test = torch.sqrt(mse_test)
print(f'Epoch [{epoch+1}/{num_epochs}] Test RMSE: {rmse_test:.4f}')

In [ ]:
show_compare_plot(preds_test.detach().numpy(), y_test_tensor.numpy())

### Промежуточные выводы

Метрика RMSE на тестовых данных не превысила пороговое значение 4500К и составила 4169.2905.

Дополнительно был построен график Факт-Прогноз для сравнения полученных значений целевого признака Температура с реальными данными.

## Выводы

В рамках данной работы с помощью нейросети была решена задача определения температуры на поверхности обнаруженных звёзд. В качестве метрики оценки качества модели использована RMSE. На тестовых данных значение метрики не превысило пороговое значение 4500 и составило 3998.1936. Результаты представлены в виде графика Факт-Прогноз для тестовых данных.

Дополнительно в рамках работы было выполнено следующее:
- предварительная загрузка и предобработка данных
- исследовательский анализ количественных и категориальных признаков
- проведен корреляционный анализ признаков
- подготовка данных для обучения нейронной сети
- разработана архитектура нейронной сети
- проведен подбор параметров сети
- для улучшения предсказаний была проведена аугментация данных